In [4]:
import os
import torch

from tqdm import tqdm

from stock_gpt import StockGPT, NaiveGPT
from dataloader_builder import build_dataloaders
from setup import Stock_GPT_cfg
from setup import path_data_preprocessor

In [5]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


In [6]:
def batch_loss(x, y, model, device, fn):
    """
    Assume x and y are on the correct device already
    """
    x = model(x)
    y_norm = (y - model.target_mean) / model.target_std
    return fn(x, y_norm)

def loader_loss(data_loader, model, device, fn, max_batches = float("inf"), pbar = None, desc=""):
    avg_loss, num_batches = 0, min(len(data_loader), max_batches)
    sub_bar = tqdm(enumerate(data_loader), total=max_batches,
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}")
    sub_bar.set_description(f"{desc} ({sub_bar.n}/{sub_bar.total})".ljust(80))
    for i, (p, t) in sub_bar:
        p = p.to(device, non_blocking=True)
        t = t.to(device, non_blocking=True)
        if i == num_batches:
            break
        avg_loss += (batch_loss(p, t, model, device, fn) - avg_loss)/(i+1)
    sub_bar.close()
    return avg_loss

In [7]:
def train_model_cuda(model, device, optimizer, max_epochs, train_dl, val_dl, fn, eval_bs, cuda_scaler):
    if os.path.exists(model.save_path):
        checkpoint = torch.load(model.save_path, map_location=device)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        cuda_scaler.load_state_dict(checkpoint["cuda_scaler"])
        epoch, train_losses, val_losses = (
            checkpoint["epoch"], checkpoint["train_losses"], checkpoint["val_losses"]
        )
    else:
        epoch, train_losses, val_losses = 0, [], []

    pbar = tqdm(total=(max_epochs-epoch)*len(train_dl), desc=f"Setting up...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}")
    try:
        for epoch in range(epoch, max_epochs):
            model.train() # sets it to training mode
            for x, y in train_dl:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    loss = batch_loss(x, y, model, device, fn)
                cuda_scaler.scale(loss).backward()
                cuda_scaler.step(optimizer)
                cuda_scaler.update()

                pbar.update(1)
                if (pbar.n % int(pbar.total*0.001)==0):
                    pbar.set_description(f"Training the model... [{pbar.n}/{pbar.total}]")
            model.eval()
            with torch.no_grad():
                pbar.set_description(f"Evaluating Epoch {epoch}... [{pbar.n}/{pbar.total}]")
                train_loss = loader_loss(train_dl, model, device, fn, eval_bs, pbar,
                                         desc="Evaluating model on training data...")
                val_loss = loader_loss(val_dl, model, device, fn,eval_bs, pbar,
                                       desc="Evaluating model on validation data...")
                pbar.write(f"Epoch {epoch}:\nTraining Loss = {train_loss}\nValidation Loss = {val_loss}")

            train_losses.append(train_loss)
            val_losses.append(val_loss)
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "cuda_scaler": cuda_scaler.state_dict(),
                "epoch": epoch,
                "train_losses": train_losses,
                "val_losses": val_losses,
            }, model.save_path)
    finally:
            pbar.close()
    return train_losses, val_losses

In [ ]:
torch.manual_seed(1234)
train_dl, val_dl, test_dl, train_norms = build_dataloaders(path_data_preprocessor)

model = StockGPT(Stock_GPT_cfg, train_norms)
total_params = sum(p.numel() for p in model.parameters())
print(total_params)
model.to(device)

naive_model = NaiveGPT(Stock_GPT_cfg, train_norms)
naive_model.to(device)

optimizer1 = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
optimizer2 = torch.optim.AdamW(naive_model.parameters(), lr=0.0004, weight_decay=0.1)
fn = torch.nn.HuberLoss()

max_epochs = 10
eval_bs = 1000
scaler = torch.amp.GradScaler("cuda")
model_train_losses, model_val_losses = train_model_cuda(model, device, optimizer1, max_epochs, 
                                                        train_dl, val_dl, fn, eval_bs, scaler)
naive_train_losses, naive_val_losses = train_model_cuda(naive_model, device, optimizer2, max_epochs,
                                                        train_dl, val_dl, fn, eval_bs, scaler)
torch.save(model.state_dict(), "stock_gpt_v1")

Reading source path at preprocessed_data/data_15min_2025.parquet...
Building DataLoaders...
3169024


|          | 0.0% (00:00) Setting up...                                                                   

## Visualize Loss

In [ ]:
torch.save(model.state_dict(), "stock_gpt_v1")

import matplotlib

In [ ]:
print(len(train_dl))
pbar = tqdm(train_dl, total=len(train_dl), desc=f"[{0}/{len(train_dl)}] Entering training data...".ljust(80),
            bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}")

res = ""
for input_batch, target_batch in iter(train_dl):
    sample = model(input_batch)
    pbar.update(1)
    pbar.set_description_str((f"[{pbar.n}/{pbar.total}] "
                              f"Entering training data...".ljust(80)))
    res = sample
    break
print(res)
print(res.shape)

test_predict = torch.randn(1, 10, 12)
out = model(test_predict)
out = out[0][-1]
print(out)
print(out.shape)